# S6E9 / HiRGE-Net + doubly nested MaxCV stack

This notebook fits the original **Hierarchical Reliability-Gated Residual Experts** network and blends it with an attached leak-safe OOF library. HiRGE uses nested target-rate priors, per-row support gating, four semantic residual experts and an eight-member neural head.

Local reference scores on common folds:

- HiRGE-Net: **0.942010** OOF ROC AUC (best local DL)
- CatBoost: **0.945322**
- doubly nested MaxCV stack: **0.945508**

The stack selects model subset, prediction representation and regularization inside each held-out meta-fold. Public leaderboard scores are not used for fitting.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

INPUT_ROOT = Path('/kaggle/input')
LIB_CANDIDATES = [
    *sorted(INPUT_ROOT.glob('s6e9-complete-oof-npy-hirge*')),
    INPUT_ROOT / 'datasets' / 'tamerlanomralinov' / 's6e9-complete-oof-npy-hirge',
]
LIB_DIR = next((path for path in LIB_CANDIDATES if (path / 'oof_hirge.npy').exists()), None)
if LIB_DIR is None:
    LIB_DIR = next(
        (path.parent for path in INPUT_ROOT.rglob('manifest.json') if (path.parent / 'oof_hirge.npy').exists()),
        None,
    )
assert LIB_DIR is not None, f'Prediction dataset not found. Mounted roots: {[str(p) for p in INPUT_ROOT.iterdir()]}'

COMP_CANDIDATES = [
    INPUT_ROOT / 'playground-series-s6e9',
    INPUT_ROOT / 'competitions' / 'playground-series-s6e9',
]
COMP_DIR = next((path for path in COMP_CANDIDATES if (path / 'train.csv').exists()), None)
if COMP_DIR is None:
    for train_path in INPUT_ROOT.rglob('train.csv'):
        if not (train_path.parent / 'test.csv').exists():
            continue
        columns = set(pd.read_csv(train_path, nrows=1).columns)
        if {'id', 'Will_Buy_EV', 'Annual_Income_USD'} <= columns:
            COMP_DIR = train_path.parent
            break
assert COMP_DIR is not None, f'Competition input not found. Mounted roots: {[str(p) for p in INPUT_ROOT.iterdir()]}'
WORK = Path('/kaggle/working')
HIRGE_OUT = WORK / 'hirge_models'
STACK_LIBRARY = WORK / 'stack_library'
STACK_OUT = WORK / 'stack_output'

# Set False for a quick audit-only rerun using the attached HiRGE predictions.
TRAIN_HIRGE = True
EPOCHS = 15
PATIENCE = 4
BATCH_SIZE = 4096

print('competition:', COMP_DIR)
print('prediction/source dataset:', LIB_DIR)
print('GPU training enabled:', TRAIN_HIRGE)
print('dataset files:', len(list(LIB_DIR.iterdir())))

## 1. Validate attached arrays and common folds

In [ ]:
train = pd.read_csv(COMP_DIR / 'train.csv', usecols=['Will_Buy_EV'])
test = pd.read_csv(COMP_DIR / 'test.csv', usecols=['id'])
y = (train['Will_Buy_EV'] == 'Yes').to_numpy(np.int8)
fold_id = np.load(LIB_DIR / 'fold_id.npy')
assert len(y) == 668_665 and len(test) == 286_571
assert fold_id.shape == (len(y),) and set(np.unique(fold_id)) == set(range(5))

BASE_NAMES = [
    'cat_native', 'conditional_moe', 'ft_transformer', 'gated_crossnet',
    'hirge', 'lgb_raw', 'lgb_te', 'offset_residual', 'tabm_lite',
    'tabm_te', 'xgb_freq', 'xgb_te'
]
rows = []
for name in BASE_NAMES:
    oof_path, test_path = LIB_DIR / f'oof_{name}.npy', LIB_DIR / f'test_{name}.npy'
    assert oof_path.exists() and test_path.exists(), name
    oof, pred = np.load(oof_path), np.load(test_path)
    assert oof.shape == (len(y),) and pred.shape == (len(test),)
    assert np.isfinite(oof).all() and np.isfinite(pred).all()
    rows.append({'model': name, 'oof_auc': roc_auc_score(y, oof)})
pd.DataFrame(rows).sort_values('oof_auc', ascending=False)

## 2. Fit the original HiRGE-Net

Target statistics are leak-safe: validation and test encodings use only the outer training fold; encodings for neural training rows are generated by a second five-fold split inside that outer fold. The network learns a reliability gate from group support and a bounded residual above the selected prior.

In [ ]:
if TRAIN_HIRGE:
    command = [
        sys.executable, str(LIB_DIR / 'train_hirge.py'),
        '--data-dir', str(COMP_DIR),
        '--output-dir', str(HIRGE_OUT),
        '--folds', '0', '1', '2', '3', '4',
        '--seeds', '94017',
        '--epochs', str(EPOCHS),
        '--patience', str(PATIENCE),
        '--batch-size', str(BATCH_SIZE),
        '--pair-weight', '0.035',
        '--device', 'cuda',
    ]
    print(' '.join(command))
    subprocess.run(command, check=True)
    fresh_oof = np.load(HIRGE_OUT / 'oof' / 'oof_hirge.npy')
    print('fresh HiRGE OOF AUC:', roc_auc_score(y, fresh_oof))
else:
    print('Using attached HiRGE arrays; training skipped.')

## 3. Rebuild the base library and run doubly nested stacking

In [ ]:
(STACK_LIBRARY / 'oof').mkdir(parents=True, exist_ok=True)
(STACK_LIBRARY / 'test').mkdir(parents=True, exist_ok=True)
STACK_OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(LIB_DIR / 'fold_id.npy', STACK_LIBRARY / 'fold_id.npy')

for name in BASE_NAMES:
    source_oof = LIB_DIR / f'oof_{name}.npy'
    source_test = LIB_DIR / f'test_{name}.npy'
    if name == 'hirge' and TRAIN_HIRGE:
        source_oof = HIRGE_OUT / 'oof' / 'oof_hirge.npy'
        source_test = HIRGE_OUT / 'test' / 'test_hirge.npy'
    shutil.copy2(source_oof, STACK_LIBRARY / 'oof' / f'oof_{name}.npy')
    shutil.copy2(source_test, STACK_LIBRARY / 'test' / f'test_{name}.npy')

stack_command = [
    sys.executable, str(LIB_DIR / 'stack_maxcv.py'),
    '--data-dir', str(COMP_DIR),
    '--model-dir', str(STACK_LIBRARY),
    '--output-dir', str(STACK_OUT),
]
print(' '.join(stack_command))
subprocess.run(stack_command, check=True)

## 4. Final validation and submission

In [ ]:
metadata = json.loads((STACK_OUT / 'maxcv_stack_metadata.json').read_text())
submission = pd.read_csv(STACK_OUT / 'submission_maxcv_nested.csv')
assert list(submission.columns) == ['id', 'Will_Buy_EV']
assert submission.shape == (len(test), 2)
assert submission['id'].equals(test['id'])
assert submission['Will_Buy_EV'].between(0, 1).all()
assert np.isfinite(submission['Will_Buy_EV']).all()

final_path = WORK / 'submission.csv'
submission.to_csv(final_path, index=False)
print(json.dumps({
    'fresh_nested_oof_auc': metadata['oof_auc'],
    'delta_vs_cat': metadata['delta_vs_cat'],
    'paired_z': metadata['z'],
    'submission_rows': len(submission),
    'prediction_mean': float(submission['Will_Buy_EV'].mean()),
    'prediction_std': float(submission['Will_Buy_EV'].std()),
    'output': str(final_path),
}, indent=2))
submission.head()

## Notes

- The `.npy` input dataset contains no competition labels or raw competition rows.
- Only the attached base OOF vectors share the exact `fold_id.npy` split.
- Blend-level OOF arrays are intentionally excluded from the fresh stack inputs to avoid second-level leakage.
- The final artifact is `/kaggle/working/submission.csv`.